# Notebook 3: Binning & Discretization

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 45 minutes

## Learning Objectives
By the end of this notebook you will be able to:
- Implement **equal-width** and **equal-frequency** binning from scratch and verify against scikit-learn
- Use **decision-tree-based binning** to find optimal, target-informed cut-points
- Explain **when discretization helps** (linear models, non-linear relationships) and **when it hurts** (tree-based models)
- Handle outliers with **clipping + binning** before modelling
- Choose between **ordinal** and **one-hot** encoding for ordered bins

**Dataset context:** Throughout this notebook we use a synthetic Airbnb-style listing dataset with features like `distance_to_centre`, `price`, `review_score`, `num_bedrooms`, and `host_experience_days`.

## The Core Analogy

Imagine you're reading a thermometer. The exact reading is **17.43°C**. For most practical purposes, you only need to know: *"Is it cold enough for a coat?"* So you mentally bin that number:

| Temperature | Category |
|------------|----------|
| below 0°C  | Freezing |
| 0 – 15°C   | Cold     |
| 15 – 25°C  | Mild     |
| above 25°C | Hot      |

17.43°C → **Mild**. You have *lost precision*, but you've gained **robustness**. An outlier of 50°C simply becomes *"Hot"* — it no longer distorts a linear model that might try to extrapolate far beyond the training range.

**Binning in ML is exactly this trade-off:** swap fine-grained numeric precision for categorical robustness, enabling linear models to capture non-linear patterns and making your features immune to extreme values.

## Why Does This Matter for ML?

1. **Outlier handling:** A listing 500 km from the city centre is just `bin=4 (far)` — it doesn't shatter a linear model's weights.
2. **Non-linearity in linear models:** Price may be flat from 0–2 km, then drop sharply from 2–5 km. Binning lets a linear model fit a step function — one bin coefficient per range.
3. **Noise reduction:** Tiny differences (3.1 km vs 3.2 km) disappear inside the same bin, reducing overfitting.
4. **Feature interactions:** `distance_bin × room_type` creates meaningful compound features (e.g., *central private room*) that are hard to express numerically.
5. **Interpretability:** Stakeholders understand "this listing is in the *high price* bin" far more easily than a raw coefficient on a log-transformed continuous variable.

In [ ]:
# ── Imports & reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import KBinsDiscretizer, OrdinalEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

np.random.seed(42)                        # reproducibility for all cells
sns.set_theme(style='whitegrid')          # consistent plot aesthetics

# ── Generate synthetic Airbnb-style dataset ────────────────────────────────
n = 2000

np.random.seed(42)
distance      = np.abs(np.random.exponential(scale=3.0, size=n))          # km, right-skewed
num_bedrooms  = np.random.choice([1, 2, 3, 4, 5], size=n,
                                  p=[0.35, 0.30, 0.20, 0.10, 0.05])
host_exp_days = np.random.exponential(scale=400, size=n)                   # days hosting

# Price has a non-linear relationship with distance: close = expensive, far = cheap
base_price    = 200 - 15 * distance + 2 * distance**2 - 0.1 * distance**3
price         = base_price + 30 * num_bedrooms + np.random.normal(0, 25, n)
price         = np.clip(price, 20, 600)                                    # realistic range

# Review scores: mostly 4-5 stars, a few bad outliers near 1
review_score  = np.clip(np.random.normal(4.5, 0.4, n), 1, 5)
outlier_idx   = np.random.choice(n, size=30, replace=False)
review_score[outlier_idx] = np.random.uniform(1, 2, 30)                   # inject bad outliers

df = pd.DataFrame({
    'distance_to_centre': distance,
    'num_bedrooms':       num_bedrooms,
    'host_experience_days': host_exp_days,
    'review_score':       review_score,
    'price':              price,
})

print(f"Dataset shape: {df.shape}")
df.describe().round(2)

## 1. Equal-Width Binning

**The idea:** Divide the range `[min, max]` into `k` intervals of identical width.

```
bin_width = (max - min) / k
edges = [min, min+w, min+2w, ..., max]
```

**Pros:** Intuitive, easy to explain.  
**Cons:** Skewed data → some bins get almost all the points, others are nearly empty. For `distance_to_centre` (exponential distribution), the far bins will be sparse.

In [ ]:
np.random.seed(42)

def equal_width_bins(x, k):
    """Assign each value in x to one of k equal-width bins.
    Returns bin labels (0-indexed) and the bin edge array.
    """
    min_val, max_val = x.min(), x.max()
    bin_edges = np.linspace(min_val, max_val, k + 1)    # k+1 edges define k bins
    # np.digitize returns which bin each value falls into;
    # we exclude the first and last edge so values at exactly min/max stay in range
    labels = np.digitize(x, bin_edges[1:-1])            # 0-indexed: 0 = leftmost bin
    return labels, bin_edges

k = 5
x = df['distance_to_centre'].values
ew_labels, ew_edges = equal_width_bins(x, k)

# ── Plot histogram with bin boundaries overlaid ────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(x, bins=60, color='steelblue', alpha=0.7, label='Raw distance')
for edge in ew_edges[1:-1]:                             # draw vertical lines at each cut
    ax.axvline(edge, color='crimson', linewidth=1.8, linestyle='--')
ax.axvline(ew_edges[1], color='crimson', linewidth=1.8, linestyle='--',
           label='Equal-width cut-points')
ax.set_xlabel('Distance to Centre (km)')
ax.set_ylabel('Count')
ax.set_title('Equal-Width Binning — distance_to_centre')
ax.legend()
plt.tight_layout()
plt.show()

# ── Bin frequency table ────────────────────────────────────────────────────
bin_freq = pd.Series(ew_labels).value_counts().sort_index()
print("\nEqual-width bin frequencies (bin index → count):")
print(bin_freq.to_string())
print("\nNote: bins 3 and 4 are sparse because distance is right-skewed.")

## 2. Equal-Frequency (Quantile) Binning

**The idea:** Instead of equal *width*, create bins with equal *count* — each bin contains roughly `n/k` data points.

We compute the `0th, 20th, 40th, 60th, 80th, 100th` percentiles as edges (for k=5). This is robust to skewed distributions because every bin is well-populated.

**Caveat:** Tied values (e.g., many zeros) can cause duplicate edges — we remove duplicates with `np.unique`.

In [ ]:
np.random.seed(42)

def equal_frequency_bins(x, k):
    """Assign each value in x to one of k equal-frequency (quantile) bins.
    Returns bin labels (0-indexed) and the quantile edge array.
    """
    quantiles = np.percentile(x, np.linspace(0, 100, k + 1))  # k+1 quantile boundaries
    quantiles = np.unique(quantiles)   # remove duplicates caused by tied values
    labels = np.digitize(x, quantiles[1:-1])                   # assign each point to a bin
    return labels, quantiles

ef_labels, ef_edges = equal_frequency_bins(x, k)

# ── Side-by-side comparison ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

# Left: equal-width
axes[0].hist(x, bins=60, color='steelblue', alpha=0.7)
for edge in ew_edges[1:-1]:
    axes[0].axvline(edge, color='crimson', linewidth=1.8, linestyle='--')
axes[0].set_title('Equal-Width Binning')
axes[0].set_xlabel('Distance to Centre (km)')
axes[0].set_ylabel('Count')

# Right: equal-frequency
axes[1].hist(x, bins=60, color='seagreen', alpha=0.7)
for edge in ef_edges[1:-1]:
    axes[1].axvline(edge, color='darkorange', linewidth=1.8, linestyle='--')
axes[1].set_title('Equal-Frequency Binning')
axes[1].set_xlabel('Distance to Centre (km)')

plt.suptitle('Equal-Width vs Equal-Frequency — distance_to_centre', fontsize=13)
plt.tight_layout()
plt.show()

# ── Frequency comparison table ─────────────────────────────────────────────
compare = pd.DataFrame({
    'EqualWidth_count':     pd.Series(ew_labels).value_counts().sort_index(),
    'EqualFrequency_count': pd.Series(ef_labels).value_counts().sort_index(),
})
print("Bin frequency comparison:")
print(compare.to_string())
print("\nEqual-frequency bins are all roughly equal (target ≈", n // k, "per bin).")

## 3. Verification Against scikit-learn `KBinsDiscretizer`

`KBinsDiscretizer` with:
- `strategy='uniform'`  → equal-width binning  
- `strategy='quantile'` → equal-frequency binning  

Our from-scratch implementations should produce identical bin assignments.

In [ ]:
np.random.seed(42)

X_col = x.reshape(-1, 1)   # KBinsDiscretizer expects a 2-D array

# ── Equal-width verification ───────────────────────────────────────────────
kbd_ew = KBinsDiscretizer(n_bins=k, encode='ordinal', strategy='uniform')
kbd_ew_labels = kbd_ew.fit_transform(X_col).ravel().astype(int)

match_ew = np.all(ew_labels == kbd_ew_labels)
print(f"Equal-width  — scratch vs sklearn match: {match_ew}")

# ── Equal-frequency verification ──────────────────────────────────────────
kbd_ef = KBinsDiscretizer(n_bins=k, encode='ordinal', strategy='quantile')
kbd_ef_labels = kbd_ef.fit_transform(X_col).ravel().astype(int)

match_ef = np.all(ef_labels == kbd_ef_labels)
print(f"Equal-freq   — scratch vs sklearn match: {match_ef}")

# Show sklearn's computed bin edges for reference
print("\nsklearn equal-width edges  :", kbd_ew.bin_edges_[0].round(2))
print("scratch equal-width edges  :", ew_edges.round(2))
print("\nsklearn equal-freq edges   :", kbd_ef.bin_edges_[0].round(2))
print("scratch equal-freq edges   :", ef_edges.round(2))

In [ ]:
np.random.seed(42)

# ── Binning a skewed target: price ─────────────────────────────────────────
# price is right-skewed; we compare equal-frequency binning vs log-transform
price_vals = df['price'].values

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Raw price
axes[0].hist(price_vals, bins=50, color='steelblue', alpha=0.8)
axes[0].set_title('Raw Price (skewed)')
axes[0].set_xlabel('Price ($)')

# Equal-frequency binning of price
price_ef_labels, _ = equal_frequency_bins(price_vals, k=5)
axes[1].bar(range(5), pd.Series(price_ef_labels).value_counts().sort_index(),
            color='seagreen', alpha=0.8)
axes[1].set_title('Equal-Freq Binned Price (5 bins)')
axes[1].set_xlabel('Bin index')
axes[1].set_ylabel('Count')

# Log transform
axes[2].hist(np.log1p(price_vals), bins=50, color='darkorange', alpha=0.8)
axes[2].set_title('log(1+price) — near-normal')
axes[2].set_xlabel('log(1+price)')

plt.suptitle('Binning vs Log-Transform for Skewed Price', fontsize=13)
plt.tight_layout()
plt.show()

print("Rule of thumb:")
print("  • Use log-transform when you want to NORMALISE a skewed numeric target/feature.")
print("  • Use binning when you want to CAPTURE NON-LINEAR STEPS in a linear model.")
print("  • For regression targets, log-transform almost always beats binning.")

## 4. Decision-Tree-Based Binning

Equal-width and equal-frequency binning are **unsupervised** — they ignore the target variable entirely. But what if the most price-informative cut-point for `distance_to_centre` is at 2.3 km, not at the 20th or 40th percentile?

**Decision-tree binning** fits a shallow regression tree on a single feature vs. the target. The tree's split thresholds are the cut-points — they are *optimal* in the sense that they maximise the reduction in prediction error.

**How it works:**
1. Fit `DecisionTreeRegressor(max_leaf_nodes=5)` using only `distance_to_centre` as input.
2. Extract the split thresholds from `tree.tree_.threshold` (sklearn stores `-2` for leaf nodes).
3. Use those thresholds as bin edges.

In [ ]:
np.random.seed(42)

def decision_tree_bins(X_col, y, max_leaf_nodes=5):
    """Find optimal bin cut-points using a shallow decision tree.
    
    Parameters
    ----------
    X_col          : 1-D array of the feature to bin
    y              : 1-D target array
    max_leaf_nodes : controls depth / number of bins produced
    
    Returns
    -------
    labels     : bin assignments for each observation
    thresholds : the split points the tree chose
    tree       : the fitted DecisionTreeRegressor (for inspection)
    """
    tree = DecisionTreeRegressor(max_leaf_nodes=max_leaf_nodes, random_state=42)
    tree.fit(X_col.reshape(-1, 1), y)                   # fit on single feature
    # tree_.threshold stores -2 for leaf nodes; keep only actual split thresholds
    raw_thresholds = tree.tree_.threshold
    thresholds = np.sort(np.unique(raw_thresholds[raw_thresholds != -2]))
    labels = np.digitize(X_col, thresholds)             # assign each point to a bin
    return labels, thresholds, tree

y = df['price'].values
dt_labels, dt_thresholds, dt_tree = decision_tree_bins(x, y, max_leaf_nodes=5)

print("Decision-tree cut-points (km):", dt_thresholds.round(3))
print("Equal-width cut-points  (km):", ew_edges[1:-1].round(3))
print("Equal-freq cut-points   (km):", ef_edges[1:-1].round(3))

# ── Plot all three sets of cut-points on the same histogram ───────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(x, bins=60, color='lightgrey', edgecolor='grey', label='Distance')

for e in ew_edges[1:-1]:
    ax.axvline(e, color='crimson',    linewidth=1.5, linestyle='--')
for e in ef_edges[1:-1]:
    ax.axvline(e, color='seagreen',   linewidth=1.5, linestyle=':')
for e in dt_thresholds:
    ax.axvline(e, color='darkorange', linewidth=2.0, linestyle='-')

# Custom legend entries
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color='crimson',    linestyle='--', label='Equal-width'),
    Line2D([0], [0], color='seagreen',   linestyle=':',  label='Equal-frequency'),
    Line2D([0], [0], color='darkorange', linestyle='-',  label='Decision-tree (optimal)'),
]
ax.legend(handles=legend_elements)
ax.set_xlabel('Distance to Centre (km)')
ax.set_ylabel('Count')
ax.set_title('Cut-point Comparison: Equal-Width vs Equal-Freq vs Decision-Tree')
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

# ── Compare model performance: raw vs. three binning strategies ────────────
# We use LinearRegression + cross-validated R² to evaluate each approach.
# Binned features must be one-hot encoded for a linear model.

from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Base feature matrix (raw numerics)
feature_cols = ['distance_to_centre', 'num_bedrooms', 'host_experience_days', 'review_score']
X_raw = df[feature_cols].values
target = df['price'].values

def ohe_bins(labels, n_bins):
    """One-hot encode bin labels into a dense array."""
    enc = OneHotEncoder(sparse_output=False, categories='auto')
    return enc.fit_transform(labels.reshape(-1, 1))

# Build feature matrices for each binning approach
other_feats = df[['num_bedrooms', 'host_experience_days', 'review_score']].values

X_ew = np.hstack([ohe_bins(ew_labels, k), other_feats])   # equal-width bins + rest
X_ef = np.hstack([ohe_bins(ef_labels, k), other_feats])   # equal-freq bins + rest
X_dt = np.hstack([ohe_bins(dt_labels, len(dt_thresholds)+1), other_feats])  # tree bins + rest

# Cross-validated R² for each approach
lr = LinearRegression()
cv_kw = dict(cv=5, scoring='r2')

r2_raw = cross_val_score(lr, X_raw, target, **cv_kw).mean()
r2_ew  = cross_val_score(lr, X_ew,  target, **cv_kw).mean()
r2_ef  = cross_val_score(lr, X_ef,  target, **cv_kw).mean()
r2_dt  = cross_val_score(lr, X_dt,  target, **cv_kw).mean()

results = pd.DataFrame({
    'Approach': ['Raw numeric', 'Equal-width bins (k=5)', 'Equal-freq bins (k=5)', 'Decision-tree bins'],
    'CV_R2':    [r2_raw, r2_ew, r2_ef, r2_dt]
})
print(results.to_string(index=False))
print("\nTree-based binning uses target information → best R² for a linear model.")
print("The non-linear distance→price relationship is better captured by informed cuts.")

## 5. Handling Outliers: Clipping + Binning

If your data contains genuine outliers (data entry errors, sensor glitches, rare events), they can **distort bin edges** — especially for equal-width binning where the range `[min, max]` drives all cut-points.

**Solution:** clip at the 5th and 95th percentiles *before* binning. This:
- Prevents outliers from creating huge, empty bins at the extremes
- Preserves the bulk of the distribution intact
- Is a defensible, auditable transformation

In [ ]:
np.random.seed(42)

# review_score has 30 injected bad outliers near 1 (simulating data errors)
rs = df['review_score'].values

p5, p95 = np.percentile(rs, 5), np.percentile(rs, 95)
rs_clipped = np.clip(rs, p5, p95)           # clip to [p5, p95] range

print(f"Before clipping — min: {rs.min():.2f}, max: {rs.max():.2f}, std: {rs.std():.3f}")
print(f"After clipping  — min: {rs_clipped.min():.2f}, max: {rs_clipped.max():.2f}, std: {rs_clipped.std():.3f}")

# ── Bin edges before vs. after clipping ───────────────────────────────────
_, edges_before = equal_width_bins(rs, k=5)
_, edges_after  = equal_width_bins(rs_clipped, k=5)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)

axes[0].hist(rs, bins=40, color='steelblue', alpha=0.7)
for e in edges_before[1:-1]:
    axes[0].axvline(e, color='crimson', linewidth=1.8, linestyle='--')
axes[0].set_title('Equal-Width Bins — Before Clipping')
axes[0].set_xlabel('Review Score')

axes[1].hist(rs_clipped, bins=40, color='seagreen', alpha=0.7)
for e in edges_after[1:-1]:
    axes[1].axvline(e, color='darkorange', linewidth=1.8, linestyle='--')
axes[1].set_title('Equal-Width Bins — After Clip(p5, p95)')
axes[1].set_xlabel('Review Score')

plt.tight_layout()
plt.show()

# ── Effect on downstream model ─────────────────────────────────────────────
labels_before, _ = equal_width_bins(rs, k=5)
labels_after,  _ = equal_width_bins(rs_clipped, k=5)

X_before = np.hstack([ohe_bins(labels_before, 5), df[['distance_to_centre', 'num_bedrooms']].values])
X_after  = np.hstack([ohe_bins(labels_after,  5), df[['distance_to_centre', 'num_bedrooms']].values])

r2_b = cross_val_score(LinearRegression(), X_before, target, cv=5, scoring='r2').mean()
r2_a = cross_val_score(LinearRegression(), X_after,  target, cv=5, scoring='r2').mean()
print(f"\nCV R² before clipping: {r2_b:.4f}")
print(f"CV R² after  clipping: {r2_a:.4f}")
print("Clipping smooths the outlier signal → more stable bin edges → better generalisation.")

## 6. When Binning Hurts: Tree-Based Models

Decision trees, Random Forests, and XGBoost **already perform binning internally** — they search for the best split threshold at each node. Pre-binning before feeding data to these models is **double discretization**: you discard information at the preprocessing stage that the tree could have used to find its own optimal splits.

**Rule:** Only bin features when using linear models, logistic regression, or other models that cannot discover non-linearities on their own.

In [ ]:
np.random.seed(42)

# ── Random Forest: raw features vs. pre-binned features ───────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Raw features (all numeric)
r2_rf_raw = cross_val_score(rf, X_raw, target, cv=5, scoring='r2').mean()

# Pre-binned distance (equal-freq) + one-hot + other raw features
r2_rf_binned = cross_val_score(rf, X_ef, target, cv=5, scoring='r2').mean()

print("Random Forest performance:")
print(f"  Raw features       CV R² = {r2_rf_raw:.4f}")
print(f"  Pre-binned features CV R² = {r2_rf_binned:.4f}")
print("\nConclusion: RF with raw features wins — trees find optimal cuts themselves.")
print("Pre-binning throws away numeric precision that the tree would have exploited.")

# ── Visual summary ─────────────────────────────────────────────────────────
methods = ['LinearReg\n(raw)', 'LinearReg\n(EW bins)', 'LinearReg\n(EF bins)',
           'LinearReg\n(DT bins)', 'RandomForest\n(raw)', 'RandomForest\n(binned)']
scores  = [r2_raw, r2_ew, r2_ef, r2_dt, r2_rf_raw, r2_rf_binned]
colors  = ['steelblue']*4 + ['seagreen', 'salmon']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(methods, scores, color=colors, edgecolor='white', linewidth=1.2)
ax.set_ylim(0, 1)
ax.set_ylabel('CV R²')
ax.set_title('Binning Helps Linear Models, Hurts Tree Models')
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{score:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

## 7. Ordinal vs. One-Hot Encoding for Bins

Once you have bins, how do you encode them?

| Encoding | When to use | Example |
|----------|-------------|--------|
| **One-hot** | Bins have no natural order, or the model should treat each bin independently | Room type: Studio, 1BR, 2BR where relative values don't matter |
| **Ordinal** | Bins have a clear, monotone order AND the model should respect it | Price tier: Low < Medium < High |

For **Linear Regression**: ordinal gives one coefficient (slope), one-hot gives k−1 coefficients. If the relationship truly is monotone, ordinal is more parsimonious.  
For **Logistic Regression**: ordinal converges faster and is less prone to overfitting when k is large.

**Warning:** Don't use ordinal encoding with tree models — they don't care about magnitude, only split values, so it makes no difference.

In [ ]:
np.random.seed(42)

# ── Create a binary classification task: is_expensive (price > median) ─────
price_median = np.median(target)
is_expensive = (target > price_median).astype(int)   # 1 = expensive, 0 = cheap

# Use equal-frequency price bins (5 bins) as the main feature
price_ef_labels, _ = equal_frequency_bins(target, k=5)

# Ordinal encoding: bins already have integer labels 0,1,2,3,4 → reshape for sklearn
X_ordinal = price_ef_labels.reshape(-1, 1).astype(float)   # shape (n, 1)

# One-hot encoding: 5 bins → 5 binary columns
ohe = OneHotEncoder(sparse_output=False)
X_onehot = ohe.fit_transform(price_ef_labels.reshape(-1, 1))  # shape (n, 5)

lr_clf = LogisticRegression(max_iter=500, random_state=42)
from sklearn.model_selection import cross_val_score

acc_ordinal = cross_val_score(lr_clf, X_ordinal, is_expensive,
                              cv=5, scoring='accuracy').mean()
acc_onehot  = cross_val_score(lr_clf, X_onehot,  is_expensive,
                              cv=5, scoring='accuracy').mean()

print(f"Logistic Regression — predicting is_expensive from price bins:")
print(f"  Ordinal encoding (1 feature)  accuracy = {acc_ordinal:.4f}")
print(f"  One-hot encoding (5 features) accuracy = {acc_onehot:.4f}")
print("\nFor ordered bins, ordinal encoding achieves similar accuracy with fewer parameters.")

# ── Show learned ordinal coefficient ──────────────────────────────────────
lr_clf.fit(X_ordinal, is_expensive)
print(f"\nOrdinal model — single coefficient: {lr_clf.coef_[0][0]:.4f}")
print("Positive coefficient confirms: higher bin → higher probability of being expensive.")

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You have a feature `property_age_years` that ranges from 1 to 120 with most values between 5 and 40 (right-skewed). You want to bin it into 6 bins for a linear model. Which binning strategy should you prefer, and why?

<details><summary>Show answer</summary>

**Equal-frequency (quantile) binning.** Because the distribution is right-skewed, equal-width bins would pack nearly all observations into the first 1–2 bins, leaving the upper bins almost empty. Equal-frequency ensures each bin contains roughly the same number of listings, so every bin contributes a useful signal to the linear model and no bin is too sparse to estimate a reliable coefficient.

</details>

---

**Question 2:** A colleague suggests pre-binning all continuous features (into 10 bins) before training an XGBoost model, arguing it speeds up training. What is the main statistical downside of this approach?

<details><summary>Show answer</summary>

**Information loss / double discretization.** XGBoost already searches for optimal split thresholds internally using a histogram approximation. By pre-binning into only 10 bins, you destroy the fine-grained numeric information that XGBoost would otherwise exploit to find more precise splits. The model's predictive performance will be worse than if you fed it raw continuous features, because you have artificially reduced the resolution of the input — the "optimal" cut-points you found unsupervised may not align with what XGBoost would have discovered using the target variable.

</details>

---

**Question 3:** You are engineering features for a linear regression model to predict Airbnb price. You create 5 equal-frequency price tier bins (Low, Medium-Low, Medium, Medium-High, High) from `distance_to_centre`. Should you apply ordinal or one-hot encoding? Does the answer change if you use a Random Forest instead?

<details><summary>Show answer</summary>

**For linear regression:** Either can work, but ordinal is reasonable if the effect of distance on price is *monotone* (e.g., further = always cheaper). It is more parsimonious (1 feature vs. 4 dummies). If the relationship is non-monotone (e.g., very central AND very far listings are cheap, middle distance is expensive), one-hot is safer because it allows the model to assign independent coefficients to each bin.  

**For Random Forest:** The choice makes very little difference. Trees split on threshold values, so ordinal (0,1,2,3,4) and one-hot produce essentially equivalent information. In practice, feeding raw continuous `distance_to_centre` to the Random Forest is better than either — it preserves full numeric resolution and lets the tree find its own splits.

</details>

## Key Takeaways

| Scenario | Recommended approach |
|----------|---------------------|
| Skewed feature, linear model | **Equal-frequency (quantile) binning** |
| Non-linear feature–target relationship, linear model | **Decision-tree binning** (uses target signal) |
| Outliers distorting bin edges | **Clip(p5, p95) first, then bin** |
| Tree-based model (RF, XGBoost) | **Do NOT bin** — feed raw numerics |
| Ordered bins (Low < Medium < High) | **Ordinal encoding** (fewer parameters) |
| Unordered / nominal bins | **One-hot encoding** |
| Skewed continuous target | **Log-transform** is usually better than binning |

**The golden rule:** Binning trades precision for robustness. Use it deliberately — only when you have a specific reason (outlier sensitivity, non-linearity in a linear model, interpretability) — not as a default preprocessing step.